# Ночная серия улучшений OSNet — Run All

Один запуск: контроль скорости → локальные признаки → кандидаты/отказ → автоматические маски → 5 обучающих пилотов → автоматическое подтверждение перспективных вариантов.

**Новый эксперимент variant19. MVP, старые notebook/runs, исходные bbox и validation не меняются. Outer-оценки и автопродвижения нет.**

Открой ядро `.venv` проекта, **Restart Kernel → Run All**. Оставь компьютер подключённым к питанию и запрети сон на время расчёта. Интернет не нужен; новые веса не скачиваются.


In [1]:
from pathlib import Path
import os
import sys

candidates = [Path.cwd(), Path.cwd() / 'Car-classification-MSK', *Path.cwd().parents]
REPO = next((p for p in candidates if (p / 'training/overnight_system.py').is_file()), None)
if REPO is None:
    raise RuntimeError('Открой notebook из проекта Car-classification-MSK')
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')

import torch
from IPython.display import Markdown, display
from training import overnight_system as experiment
from training.overnight_training import training_jobs
print('Repository:', REPO, '| Torch:', torch.__version__)


Repository: /Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK | Torch: 2.14.0


## Настройки

По умолчанию всё включено; менять `PHASE` не нужно. `WALL_HOURS = None` означает **без общего ограничения времени: работать до конца очереди**. Завершённые задачи при повторном Run All не пересчитываются. Остальные настройки заморожены в manifest; для их изменения нужен новый `RUN_NAME`.

Обучения идут последовательно на том же устройстве, что v16. Они не запускаются одновременно на одной GPU/MPS. Внутренние сравнения — по шагам оптимизатора, а не по эпохам разной длины.


In [2]:
RUN_NAME = 'overnight_v1'
SOURCE_RUN = 'review_v1'
POLICY_RUN = 'policy_v1'
WALL_HOURS = None  # Без ограничения времени: выполнить всю очередь
RUN_TRAINING = True
AUTO_CONFIRM = True
RUN_MASK_DIAGNOSTICS = True
RUN_BENCHMARK = True
ALLOW_CPU_TRAINING = False

source_manifest = experiment.read(experiment.review.VARIANT / 'runs' / SOURCE_RUN / 'manifest.json')
torch.set_num_threads(source_manifest['runtime']['torch_threads'])
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.use_deterministic_algorithms(source_manifest['runtime']['deterministic'], warn_only=True)
print('Training device:', source_manifest['runtime']['device'])
print('Time limit:', 'disabled — run until complete' if WALL_HOURS is None else f'{WALL_HOURS} hours')
print('Outer evaluation: disabled; MVP promotion: disabled')
print('Local policies:', len(experiment.evaluation.LOCAL_GRID), '(12 new + baseline), 3 seeds, 3 primary episodes')
for name, job in training_jobs().items():
    print(f'{name}: stop={job.stop_step}, LR horizon={job.lr_horizon}, CE scale={job.ce_scale:.3f}, KD={job.relation_weight}')


Training device: mps
Time limit: disabled — run until complete
Outer evaluation: disabled; MVP promotion: disabled
Local policies: 13 (12 new + baseline), 3 seeds, 3 primary episodes
R1_control: stop=800, LR horizon=1700, CE scale=1.000, KD=0.0
K2_ce1: stop=800, LR horizon=1700, CE scale=1.000, KD=0.0
K2_ce_sqrt544: stop=800, LR horizon=1700, CE scale=23.324, KD=0.0
R1_kd01: stop=800, LR horizon=1700, CE scale=1.000, KD=0.1
R1_kd1: stop=800, LR horizon=1700, CE scale=1.000, KD=1.0


## Предварительная проверка

Проверяются источник v16/v18, сохранённые веса, код, split, изображения и исходные аннотации. Повреждение или несовместимость исходных артефактов останавливает серию. Эта ячейка ничего не обучает.


In [3]:
context = experiment.prepare(
    RUN_NAME, source_run=SOURCE_RUN, policy_run=POLICY_RUN,
    training=RUN_TRAINING, confirmation=AUTO_CONFIRM,
    masks=RUN_MASK_DIAGNOSTICS, benchmark=RUN_BENCHMARK,
)
print('Output:', context['output'])
print('Frozen signature:', context['signature'])
print('Preflight passed. Следующая ячейка запускает очередь.')


Output: /Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/OSNet-AIN-x1.0/variant_19_overnight_system/runs/overnight_v1
Frozen signature: 7cfd1df47fd025b4137b3c67ab07f18c2a7827553b4e0b434d2bfc5e8a242929
Preflight passed. Следующая ячейка запускает очередь.


## Ночной запуск

Печатаются текущая задача, время, шаги обучения и ETA каждые 25 шагов. Каждый завершённый эксперимент сохраняется отдельно. При обычной ошибке независимые задачи продолжаются; ошибки целостности останавливают всю серию. Прерывание возобновляется с последнего сохранённого блока; незавершённый блок будет повторён.

Общего лимита времени нет: очередь выполняется до конца. **После сбоя для продолжения: тот же `RUN_NAME` → Restart Kernel → Run All. Не удаляй `runs`.** Если позже явно включить числовой лимит, обучение приостановится на сохранённой границе блока; прочие задачи закончат текущую единицу работы.

Автоотбор: пилот должен дать ≥0,3 п.п. относительно своего контроля и R1; затем 3 seed должны дать среднее ≥0,2 п.п. и положительный знак в ≥2/3 запусках. Только после этого запускается alternate. Это фильтр затрат, не доказательство статистической значимости.


In [4]:
result = experiment.run(context, wall_hours=WALL_HOURS, allow_cpu_training=ALLOW_CPU_TRAINING)
print('Status:', result['status'])
print('Outer evaluated:', result['outer_evaluated'], '| MVP promoted:', result['promoted'])
print('Original artifacts unchanged:', result.get('protected_unchanged'))
display(Markdown((context['output'] / 'REPORT.md').read_text(encoding='utf-8')))


[START] benchmark | elapsed 0.00 h
[DONE] benchmark | 5.4 min
[START] local_primary_20260915 | elapsed 0.09 h
  local features primary/20260915/original: 32/1036
  local features primary/20260915/original: 352/1036
  local features primary/20260915/original: 672/1036
  local features primary/20260915/original: 992/1036
  primary/20260915/regular_20260915: 13 local policies evaluated
  primary/20260915/regular_20261016: 13 local policies evaluated
  primary/20260915/regular_20261117: 13 local policies evaluated
[DONE] local_primary_20260915 | 0.8 min
[START] local_primary_20260916 | elapsed 0.10 h
  local features primary/20260916/original: 32/1036
  local features primary/20260916/original: 352/1036
  local features primary/20260916/original: 672/1036
  local features primary/20260916/original: 992/1036
  primary/20260916/regular_20260915: 13 local policies evaluated
  primary/20260916/regular_20261016: 13 local policies evaluated
  primary/20260916/regular_20261117: 13 local policies 

/Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/.venv/lib/python3.11/site-packages/torch/autograd/graph.py:1072: UserWarning: index_put_with_accumulate_mps does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /Users/runner/work/pytorch/pytorch/aten/src/ATen/Context.cpp:190.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


primary/R1_kd01/20260915: 25/800 | loss=8.0864 | elapsed 00:00:41 | ETA 00:21:21
primary/R1_kd01/20260915: 50/800 | loss=7.5212 | elapsed 00:01:22 | ETA 00:20:36
primary/R1_kd01/20260915: 75/800 | loss=6.3452 | elapsed 00:02:03 | ETA 00:19:52
primary/R1_kd01/20260915: 100/800 | loss=6.2891 | elapsed 00:02:44 | ETA 00:19:08
primary/R1_kd01/20260915: 125/800 | loss=5.3815 | elapsed 00:03:25 | ETA 00:18:27
primary/R1_kd01/20260915: 150/800 | loss=3.5492 | elapsed 00:04:06 | ETA 00:17:47
primary/R1_kd01/20260915: 175/800 | loss=3.8446 | elapsed 00:04:47 | ETA 00:17:05
primary/R1_kd01/20260915: 200/800 | loss=3.5003 | elapsed 00:05:28 | ETA 00:16:24
primary/R1_kd01/20260915: 225/800 | loss=2.7709 | elapsed 00:06:09 | ETA 00:15:43
primary/R1_kd01/20260915: 250/800 | loss=1.9207 | elapsed 00:06:50 | ETA 00:15:02
primary/R1_kd01/20260915: 275/800 | loss=1.7683 | elapsed 00:07:31 | ETA 00:14:21
primary/R1_kd01/20260915: 285/800 | loss=2.1523 | elapsed 00:07:48 | ETA 00:14:05
primary/R1_kd01/202

# Ночная серия OSNet / variant19

Статус: **complete**.
Outer validation не оценивалась. Автоматического переключения MVP нет.
Сохранность исходной разметки/MVP/старых результатов подтверждена SHA256.

## Задачи

| Задача | Статус |
|---|---|
| benchmark | complete |
| local_primary_20260915 | complete |
| local_primary_20260916 | complete |
| local_primary_20260917 | complete |
| local_selection | complete |
| local_alternate_20260915 | complete |
| local_alternate_20260916 | complete |
| local_alternate_20260917 | complete |
| confidence_primary | complete |
| confidence_selection | complete |
| confidence_alternate | complete |
| automatic_mask_query | complete |
| automatic_mask_gallery | complete |
| automatic_mask_both | complete |
| train_primary_R1_control_20260915 | complete |
| evaluate_primary_R1_control_20260915 | complete |
| train_primary_K2_ce1_20260915 | complete |
| evaluate_primary_K2_ce1_20260915 | complete |
| train_primary_K2_ce_sqrt544_20260915 | complete |
| evaluate_primary_K2_ce_sqrt544_20260915 | complete |
| train_primary_R1_kd01_20260915 | complete |
| evaluate_primary_R1_kd01_20260915 | complete |
| train_primary_R1_kd1_20260915 | complete |
| evaluate_primary_R1_kd1_20260915 | complete |
| training_pilot_selection | complete |

## Локальный scorer: primary

| Настройка | Средний mAP@10 |
|---|---:|
| baseline | 0.84295 |
| mean_best_k20_w05 | 0.84266 |
| mean_best_k20_w10 | 0.84201 |
| mean_best_k20_w20 | 0.84102 |
| mean_best_k50_w05 | 0.84266 |
| mean_best_k50_w10 | 0.84201 |
| mean_best_k50_w20 | 0.84102 |
| partial_k20_w05 | 0.84260 |
| partial_k20_w10 | 0.84260 |
| partial_k20_w20 | 0.84192 |
| partial_k50_w05 | 0.84260 |
| partial_k50_w10 | 0.84260 |
| partial_k50_w20 | 0.84192 |

Замороженный выбор для alternate: `baseline`.
Alternate: среднее изменение +0.000 п.п.; gate=False.
Парный bootstrap по identity: {'identities': 182, 'mean_delta': 0.0, 'interval': [0.0, 0.0], 'note': 'conditional identity-cluster bootstrap, not independent training-seed uncertainty'}

## Кандидаты: primary, отдельные held-out identity

| Политика | F1 | TNR | C | Coverage | Ошибки среди принятых |
|---|---:|---:|---:|---:|---:|
| R1_first_seed/cosine/raw_top1 | 0.8333 | 0.8333 | 0.8333 | 0.6739 | 0.1129 |
| R1_first_seed/cosine/ranking_top1 | 0.8333 | 0.8333 | 0.8333 | 0.6739 | 0.1129 |
| R1_first_seed/cosine_plus_margin/raw_top1 | 0.8244 | 0.8333 | 0.8271 | 0.6630 | 0.1148 |
| R1_first_seed/cosine_plus_margin/ranking_top1 | 0.8244 | 0.8333 | 0.8271 | 0.6630 | 0.1148 |
| R1_first_seed/cosine_plus_local/raw_top1 | 0.8244 | 0.8333 | 0.8271 | 0.6630 | 0.1148 |
| R1_first_seed/cosine_plus_local/ranking_top1 | 0.8244 | 0.8333 | 0.8271 | 0.6630 | 0.1148 |
| R1_equal3/cosine/raw_top1 | 0.8759 | 0.8333 | 0.8631 | 0.7283 | 0.1045 |
| R1_equal3/cosine/ranking_top1 | 0.8759 | 0.8333 | 0.8631 | 0.7283 | 0.1045 |
| R1_equal3/cosine_plus_margin/raw_top1 | 0.8759 | 0.8333 | 0.8631 | 0.7283 | 0.1045 |
| R1_equal3/cosine_plus_margin/ranking_top1 | 0.8759 | 0.8333 | 0.8631 | 0.7283 | 0.1045 |
| R1_equal3/cosine_minus_disagreement/raw_top1 | 0.8593 | 0.8333 | 0.8515 | 0.7065 | 0.1077 |
| R1_equal3/cosine_minus_disagreement/ranking_top1 | 0.8593 | 0.8333 | 0.8515 | 0.7065 | 0.1077 |

## Кандидаты: alternate, отдельные held-out identity

| Политика | F1 | TNR | C | Coverage | Ошибки среди принятых |
|---|---:|---:|---:|---:|---:|
| R1_first_seed/cosine/raw_top1 | 0.7752 | 0.7222 | 0.7593 | 0.6739 | 0.1935 |
| R1_first_seed/cosine/ranking_top1 | 0.7656 | 0.7222 | 0.7526 | 0.6739 | 0.2097 |
| R1_equal3/cosine/raw_top1 | 0.8030 | 0.7222 | 0.7788 | 0.6957 | 0.1719 |
| R1_equal3/cosine/ranking_top1 | 0.7939 | 0.7222 | 0.7724 | 0.6957 | 0.1875 |

## Обучение: односидовые пилоты

| Рецепт | mAP@10 primary |
|---|---:|
| R1_control | 0.84410 |
| K2_ce1 | 0.72708 |
| K2_ce_sqrt544 | 0.76682 |
| R1_kd01 | 0.83875 |
| R1_kd1 | 0.83293 |

Отбор пилотов: {'selected': [], 'families': {'ce': {'candidate': 'K2_ce_sqrt544', 'control': 'K2_ce1', 'gain': 0.03974513203679875, 'gain_over_r1': -0.07727672613089276, 'passed': False}, 'distillation': {'candidate': 'R1_kd01', 'control': 'R1_control', 'gain': -0.005349337432670853, 'gain_over_r1': -0.005349337432670853, 'passed': False}}, 'minimum_gain': 0.003, 'scope': 'one-seed pilot, not a final win'}

## Границы вывода

- Все метрики новых вариантов относятся к held-out train IDs; это не новая outer-метрика.
- Локальный scorer заморожен и не обучает pair-head. DINO/обучаемая OOF-голова не запускались.
- Автоматические маски — proxy, не plate-only аудит; координат подтверждённых номеров нет.
- CPU benchmark не подтверждает баллы RTX A5000/CUDA и не переключает приложение.
- Smooth-AP и GPU dependency lock отложены; веса/порог не выбираются по outer.
- По умолчанию ограничения времени нет. После прерывания повторите Run All с тем же RUN_NAME; завершённые задачи пропускаются.


## Где смотреть результаты

`REPORT.md` — сводка; `results.json` — детали; `progress.json` — завершённые/упавшие задачи; `tasks/<имя>/error.json` — причина ошибки. Чекпоинты и optimizer/RNG лежат отдельно в `training/`. Отрицательный результат тоже сохраняется.

Ограничения: локальный scorer пока замороженный, без обучаемой OOF-головы/DINO; маски автоматические, **не plate-only**; CPU-замер не подтверждает баллы GPU. Публичный test используется только для замера замороженного инференса и проверки формата, не для подбора. GPU-lock, точный аудит номера, Smooth-AP и переключение приложения не включены в ночную серию.


In [5]:
print('Summary:', context['output'] / 'REPORT.md')
print('Details:', context['output'] / 'results.json')
failed = [e for e in result['events'] if e['status'] == 'failed']
print('Failed tasks:', len(failed))
for entry in failed:
    print(entry['task'], '→', context['output'] / 'tasks' / entry['task'] / 'error.json')
if result['status'] == 'time_budget_reached':
    print('Бюджет времени исчерпан. Повтори Run All с тем же RUN_NAME для продолжения.')


Summary: /Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/OSNet-AIN-x1.0/variant_19_overnight_system/runs/overnight_v1/REPORT.md
Details: /Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/OSNet-AIN-x1.0/variant_19_overnight_system/runs/overnight_v1/results.json
Failed tasks: 0
